# Model Evaluation Notebook

This notebook evaluates a selected LoRA adapter for industrial automation next-step prediction. It loads a test dataset from `test_data.jsonl`, groups samples by total step count, and draws up to `N_PER_STEP_COUNT` examples per group for evaluation.

For each sampled example, the notebook runs autoregressive single-step inference, scores the prediction against the reference sequence with custom penalties, measures latency, and saves tables, plots, and a text report under `evaluation/`.


In [ ]:
import os
import sys
import gc
import re
import json
import time
import random
import statistics
from collections import defaultdict, Counter
from datetime import datetime
from pathlib import Path
import tkinter as tk
from tkinter import filedialog

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datasets import load_dataset
from unsloth import FastLanguageModel

# =========================
# Configuration (editable)
# =========================
BASE_MODEL = "" # Replace with your base model name
MAX_SEQ_LENGTH = 3072
DTYPE = None
LOAD_IN_4BIT = True

# Key setting: single-step prediction only needs a small number of tokens, so 64 is much faster
MAX_NEW_TOKENS = 64  
TEMPERATURE = 0.0
TOP_P = 1.0
DO_SAMPLE = False

SEED = 3407
N_PER_STEP_COUNT = 16

# Note: all evaluation samples are drawn from the same JSONL test file
TEST_FILE = "test_data.jsonl"
EVAL_DIR = Path("evaluation")

# Updated instruction focused on single-step prediction
FIXED_INSTRUCTION = (
    "You are an industrial automation planning system.\n"
    "Given the system configuration (Input JSON) and the history of executed operations [Past Steps], "
    "predict ONLY the exact SINGLE next step required to fulfill the order.\n\n"
    "Format your output exactly as:\n"
    "Step N | Op: ... | Cost: ... | Dur: ...\n\n"
    "If the order is completely fulfilled and no further actions are needed, output:\n"
    "Step N | Op: End | Cost: 0.000 | Dur: 0.000\n\n"
    "Do NOT output any extra keys, JSON, reasoning text, or multiple steps."
)

PROMPT_TEMPLATE = (
    "Below is an instruction that describes a task, paired with an input that provides further context. "
    "Write a response that appropriately completes the request.\n\n"
    "### Instruction:\n{instruction}\n\n"
    "### Input:\n{input}\n\n"
    "### Response:\n"
)

RESPONSE_MARKER = "### Response:\n"
STEP_PATTERN = re.compile(
    r"^\s*Step\s*(?P<idx>\d+)\s*\|\s*Op\s*:\s*(?P<op>.*?)\s*\|\s*Cost\s*:\s*(?P<cost>[-+]?(?:\d+(?:\.\d*)?|\.\d+))\s*\|\s*Dur\s*:\s*(?P<dur>[-+]?(?:\d+(?:\.\d*)?|\.\d+))\s*$",
    re.IGNORECASE,
)


def choose_model_dir() -> str:
    """Open a GUI folder picker and return the selected model directory."""
    root = tk.Tk()
    root.withdraw()
    root.attributes("-topmost", True)
    default_model_dir = Path.cwd().resolve().parent / "Model"
    if not default_model_dir.is_dir():
        default_model_dir = Path.cwd().resolve()
    model_dir = filedialog.askdirectory(
        title="Select LoRA model directory",
        initialdir=str(default_model_dir),
    )
    root.destroy()
    if not model_dir:
        raise RuntimeError("No model directory selected.")
    return model_dir


def resolve_adapter_dir(model_dir: str) -> str:
    """Validate adapter dir and optionally auto-resolve from a parent folder."""
    path = Path(model_dir).expanduser().resolve()
    if (path / "adapter_config.json").is_file():
        return str(path)

    candidates = []
    for child in path.iterdir():
        if child.is_dir() and (child / "adapter_config.json").is_file():
            candidates.append(child)

    if len(candidates) == 1:
        chosen = candidates[0].resolve()
        print(f"Auto-resolved adapter dir: {chosen}")
        return str(chosen)

    if len(candidates) > 1:
        names = "\n".join(str(c) for c in sorted(candidates))
        raise FileNotFoundError(
            "Selected folder does not contain adapter files directly. "
            "Multiple LoRA subfolders found, please pick one exactly:\n"
            f"{names}"
        )

    raise FileNotFoundError(
        f"No adapter_config.json found in {path}. Please choose a LoRA output folder."
    )


def resolve_base_model_name(model_dir: str, default_base_model: str) -> str:
    """Read base model name from adapter_config.json when available."""
    cfg_path = Path(model_dir) / "adapter_config.json"
    try:
        cfg = json.loads(cfg_path.read_text(encoding="utf-8"))
    except Exception as e:
        print(f"Warning: failed to read {cfg_path}: {e}")
        print(f"Fallback to BASE_MODEL: {default_base_model}")
        return default_base_model

    adapter_base = cfg.get("base_model_name_or_path")
    if adapter_base:
        print(f"Adapter base model from config: {adapter_base}")
        if adapter_base != default_base_model:
            print(f"Override BASE_MODEL ({default_base_model}) -> {adapter_base}")
        return adapter_base

    print(f"adapter_config.json has no base_model_name_or_path; use BASE_MODEL: {default_base_model}")
    return default_base_model


# Dynamically build the prompt for each step
def build_step_prompt(context_str: str, past_steps: list) -> str:
    """Build the prompt for the current step using the sliced-step format."""
    if not past_steps:
        past_str = "None"
    else:
        past_str = "\n".join(past_steps)
        
    combined_input = f"{context_str}\n\n[Past Steps]\n{past_str}\n\n[Next Step Prediction]"
    return PROMPT_TEMPLATE.format(instruction=FIXED_INSTRUCTION, input=combined_input)


def trim_generated_text(decoded: str, tokenizer) -> str:
    """Keep only assistant response text after the response marker."""
    pos = decoded.find(RESPONSE_MARKER)
    if pos != -1:
        decoded = decoded[pos + len(RESPONSE_MARKER):]
    eos = tokenizer.eos_token or ""
    if eos:
        decoded = decoded.replace(eos, "")
    return decoded.strip()


def normalize_op(op: str) -> str:
    """Normalize operation text for robust string matching."""
    return " ".join(op.strip().lower().split())


def should_relax_port_match(op_norm: str) -> bool:
    return op_norm.startswith(("connect", "dosing", "separation"))


def normalize_port_index(op_norm: str) -> str:
    # For hc20_out1 / hc30_in2 style ports, ignore trailing index number.
    return re.sub(r"\b([a-z]+\d+)_(in|out)\d+\b", r"\1_\2", op_norm)


def is_connect_op(op_norm: str) -> bool:
    return op_norm.startswith("connect")


def parse_steps(text: str):
    """Parse all step lines; return None if format is invalid."""
    lines = [ln.strip() for ln in text.splitlines() if ln.strip()]
    step_lines = [ln for ln in lines if ln.lower().startswith("step")]
    if not step_lines:
        return None

    steps = []
    for ln in step_lines:
        m = STEP_PATTERN.match(ln)
        if not m:
            return None
        try:
            cost = float(m.group("cost"))
            dur = float(m.group("dur"))
        except ValueError:
            return None
        op_raw = m.group("op").strip()
        op_norm = normalize_op(op_raw)
        op_match = normalize_port_index(op_norm) if should_relax_port_match(op_norm) else op_norm
        steps.append(
            {
                "step_idx": int(m.group("idx")),
                "op_raw": op_raw,
                "op_norm": op_norm,
                "op_match": op_match,
                "cost": cost,
                "dur": dur,
                "is_connect": is_connect_op(op_norm),
                "line": ln,
            }
        )
    return steps


def align_steps(ref_steps, pred_steps):
    """
    Order-preserving alignment with non-cascading penalties.

    - Exact op_match aligns as match.
    - Replacement (wrong predicted step) counts as one missing penalty.
    - Insertion (extra predicted step) counts as one extra penalty.
    - Deletion (missing reference step) counts as one missing penalty.
    """
    n = len(ref_steps)
    m = len(pred_steps)

    # dp[i][j] = (total_edit, replace_count)
    dp = [[(0, 0)] * (m + 1) for _ in range(n + 1)]
    bt = [[None] * (m + 1) for _ in range(n + 1)]

    for i in range(1, n + 1):
        dp[i][0] = (i, 0)
        bt[i][0] = "D"  # delete ref -> missing
    for j in range(1, m + 1):
        dp[0][j] = (j, 0)
        bt[0][j] = "I"  # insert pred -> extra

    for i in range(1, n + 1):
        for j in range(1, m + 1):
            if ref_steps[i - 1]["op_match"] == pred_steps[j - 1]["op_match"]:
                dp[i][j] = dp[i - 1][j - 1]
                bt[i][j] = "M"
                continue

            # Replacement / deletion / insertion each adds one edit;
            # secondary objective minimizes replacement count so extra/missing
            # are not swallowed into long substitution chains.
            sub = (dp[i - 1][j - 1][0] + 1, dp[i - 1][j - 1][1] + 1)
            delete = (dp[i - 1][j][0] + 1, dp[i - 1][j][1])
            insert = (dp[i][j - 1][0] + 1, dp[i][j - 1][1])

            best = min(sub, delete, insert)
            dp[i][j] = best
            if best == sub:
                bt[i][j] = "S"
            elif best == delete:
                bt[i][j] = "D"
            else:
                bt[i][j] = "I"

    i, j = n, m
    matches = []
    missing_ref = []
    extras = []
    replacements = []

    while i > 0 or j > 0:
        op = bt[i][j]
        if op == "M":
            matches.append((j - 1, i - 1))
            i -= 1
            j -= 1
        elif op == "S":
            replacements.append((j - 1, i - 1))
            i -= 1
            j -= 1
        elif op == "D":
            missing_ref.append(i - 1)
            i -= 1
        elif op == "I":
            extras.append(j - 1)
            j -= 1
        else:
            break

    matches.reverse()
    missing_ref.reverse()
    extras.reverse()
    replacements.reverse()
    return matches, missing_ref, extras, replacements


def compute_score(reference_text: str, prediction_text: str):
    """Compute score and return detailed penalty breakdown."""
    ref_steps = parse_steps(reference_text)
    pred_steps = parse_steps(prediction_text)

    if ref_steps is None or pred_steps is None:
        return {
            "score": 0.0,
            "format_ok": False,
            "total_steps": 0 if ref_steps is None else len(ref_steps),
            "replace_count": 0,
            "missing_count": 0,
            "wrong_order_count": 0,
            "cost_dur_mismatch_count": 0,
            "extra_count": 0,
        }

    total_steps = len(ref_steps)
    if total_steps == 0:
        return {
            "score": 0.0,
            "format_ok": False,
            "total_steps": 0,
            "replace_count": 0,
            "missing_count": 0,
            "wrong_order_count": 0,
            "cost_dur_mismatch_count": 0,
            "extra_count": 0,
        }

    unit = 100.0 / total_steps
    matches, missing_ref, extras, replacements = align_steps(ref_steps, pred_steps)

    missing_count = len(missing_ref)
    extra_count = len(extras)
    replace_count = len(replacements)
    wrong_order_count = 0

    cost_dur_mismatch_count = 0
    tol = 1e-6
    for pidx, ridx in matches:
        p = pred_steps[pidx]
        r = ref_steps[ridx]
        if abs(p["cost"] - r["cost"]) > tol or abs(p["dur"] - r["dur"]) > tol:
            cost_dur_mismatch_count += 1

    score = 100.0
    # Penalize wrong/missing/extra equally without cascading order penalties.
    score -= (replace_count + missing_count + extra_count) * unit * 2
    score -= cost_dur_mismatch_count * unit * 0.5
    score = max(0.0, min(100.0, score))

    return {
        "score": score,
        "format_ok": True,
        "total_steps": total_steps,
        "replace_count": replace_count,
        "missing_count": missing_count,
        "wrong_order_count": wrong_order_count,
        "cost_dur_mismatch_count": cost_dur_mismatch_count,
        "extra_count": extra_count,
    }


def load_records(file_path: str, split_name: str):
    """Load JSONL records and keep split metadata."""
    records = []
    with open(file_path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            line = line.strip()
            if not line:
                continue
            obj = json.loads(line)
            records.append(
                {
                    "split": split_name,
                    "row_id": i,
                    "input": obj["input"],
                    "output": obj["output"],
                }
            )
    return records


def build_step_groups(records):
    """Parse records and group valid samples by total step count."""
    grouped = defaultdict(list)
    invalid_count = 0
    for rec in records:
        ref_steps = parse_steps(rec["output"])
        if ref_steps is None:
            invalid_count += 1
            continue
        step_count = len(ref_steps)
        rec2 = dict(rec)
        rec2["step_count"] = step_count
        grouped[step_count].append(rec2)
    per_step_total = {k: len(v) for k, v in grouped.items()}
    return grouped, per_step_total, invalid_count


def grouped_sampling(records, n_per_group: int, rng: random.Random):
    """Sample by step-count from a single test pool, capped at n_per_group per step-count."""
    grouped, total_dist, invalid_count = build_step_groups(records)

    sampled = []
    sampled_dist = {}

    for step_count in sorted(grouped.keys()):
        pool = list(grouped[step_count])
        if len(pool) <= n_per_group:
            picked = pool
        else:
            picked = rng.sample(pool, n_per_group)

        sampled.extend(picked)
        sampled_dist[step_count] = len(picked)

    return {
        "sampled": sampled,
        "total_dist": total_dist,
        "sampled_dist": sampled_dist,
        "invalid_count": invalid_count,
    }


# Autoregressive single-step inference engine
def run_autoregressive_inference(model, tokenizer, context_str: str, max_steps=30):
    """Run autoregressive single-step generation in a loop."""
    past_steps = []
    total_latency = 0.0
    
    for step_idx in range(1, max_steps + 1):
        # 1. Build the prompt for the current step from the history
        prompt = build_step_prompt(context_str, past_steps)
        
        inputs = tokenizer(
            prompt,
            return_tensors="pt",
            add_special_tokens=False,
        ).to(model.device)

        start = time.perf_counter()
        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=DO_SAMPLE,
                temperature=TEMPERATURE,
                top_p=TOP_P,
                eos_token_id=tokenizer.eos_token_id,
                pad_token_id=tokenizer.eos_token_id,
            )
        elapsed = time.perf_counter() - start
        total_latency += elapsed

        # 2. Extract the generated text
        decoded = tokenizer.decode(out[0], skip_special_tokens=False)
        resp = trim_generated_text(decoded, tokenizer)
        
        # Use the first non-empty line as the predicted operation for this step
        lines = [ln.strip() for ln in resp.strip().split('\n') if ln.strip()]
        if not lines:
            break # The model produced no content, so stop early
            
        first_line = lines[0]
        past_steps.append(first_line)
        
        # 3. Check the stopping condition
        if "Op: End" in first_line or "Op: end" in first_line:
            break
            
    # Join all predicted single steps into a multi-line string for compatibility with compute_score
    full_prediction = "\n".join(past_steps)
    return full_prediction, total_latency


def summarize_and_save(
    df: pd.DataFrame,
    model_dir: str,
    output_dir: Path,
    base_model_name: str,
    run_now: str | None = None,
    model_output_dir: Path | None = None,
    terminal_log_path: Path | None = None,
):
    output_dir.mkdir(parents=True, exist_ok=True)

    model_name = Path(model_dir).name
    now = run_now or datetime.now().strftime("%Y%m%d_%H%M%S")
    avg_score = df["score"].mean() if len(df) else 0.0
    avg_time = df["latency_s"].mean() if len(df) else 0.0

    safe_model_name = re.sub(r"[^a-zA-Z0-9._-]+", "_", model_name)
    if model_output_dir is None:
        model_output_dir = output_dir / model_name
    model_output_dir.mkdir(parents=True, exist_ok=True)

    report_name = f"{safe_model_name}_avg{avg_score:.2f}_{now}.txt"
    csv_name = f"{safe_model_name}_details_{now}.csv"
    report_path = model_output_dir / report_name
    csv_path = model_output_dir / csv_name

    print("\n" + "=" * 80)
    print("Evaluation Summary")
    print("=" * 80)
    print(f"Model Name: {model_name}")
    print(f"Base Model: {base_model_name}")
    print(f"Model Directory: {model_dir}")
    print(f"Average Score: {avg_score:.4f}")
    print(f"Average Response Time (s): {avg_time:.4f}")
    print(f"Test Sample Count: {len(df)}")

    print("\nPer-split summary:")
    if len(df):
        split_summary = df.groupby("split").agg(
            sample_count=("score", "count"),
            avg_score=("score", "mean"),
            avg_latency_s=("latency_s", "mean"),
        )
        print(split_summary)

    print("\nStep-count sample distribution (selected):")
    step_dist = (
        df.groupby(["split", "step_count"]).size().rename("count")
        if len(df)
        else pd.Series(dtype=int)
    )
    print(step_dist)

    step_dist_table = (
        df.groupby(["split", "step_count"]).size().unstack(fill_value=0).sort_index(axis=1)
        if len(df)
        else pd.DataFrame()
    )
    step_total = step_dist_table.sum(axis=0).astype(int) if len(step_dist_table) else pd.Series(dtype=int)
    print("\nStep-count total distribution (train+val):")
    print(step_total)

    step_quality = (
        df.groupby("step_count")
        .agg(
            sample_count=("score", "count"),
            avg_score=("score", "mean"),
            avg_wrong_step=("replace_count", "mean"),
            avg_extra_step=("extra_count", "mean"),
            avg_missing=("missing_count", "mean"),
            avg_wrong_order=("wrong_order_count", "mean"),
            avg_attr_mismatch=("cost_dur_mismatch_count", "mean"),
        )
        .sort_index()
        if len(df)
        else pd.DataFrame()
    )
    print("\nStep-count quality summary (train+val combined):")
    print(step_quality)

    score_bins = [0, 20, 40, 60, 80, 90, 95, 100.0001]
    score_hist = (
        pd.cut(df["score"], bins=score_bins, right=False).value_counts().sort_index()
        if len(df)
        else pd.Series(dtype=int)
    )

    if len(df):
        lat_min = float(df["latency_s"].min())
        lat_max = float(df["latency_s"].max())
        if lat_max - lat_min < 1e-9:
            latency_bins = [lat_min, lat_min + 1e-6]
        else:
            latency_bins = np.linspace(lat_min, lat_max, 8)
        latency_hist, latency_edges = np.histogram(df["latency_s"], bins=latency_bins)
    else:
        latency_hist, latency_edges = np.array([]), np.array([])

    print("\nScore distribution (bins):")
    print(score_hist)

    print("\nLatency distribution (bins):")
    if len(latency_hist):
        for i, c in enumerate(latency_hist):
            print(f"[{latency_edges[i]:.4f}, {latency_edges[i+1]:.4f}) -> {int(c)}")

    df.to_csv(csv_path, index=False, encoding="utf-8")

    plot_paths = []

    if len(step_total):
        fig1 = plt.figure(figsize=(10, 5))
        x_vals = np.arange(len(step_total.index))
        bars0 = plt.bar(
            x_vals,
            step_total.values,
            width=0.6,
            alpha=0.9,
            color="#9ecae1",
            edgecolor="#4a4a4a",
            linewidth=0.8,
            label="total",
        )
        for b in bars0:
            h = b.get_height()
            plt.text(
                b.get_x() + b.get_width() / 2,
                h,
                f"{int(round(h))}",
                ha="center",
                va="bottom",
                fontsize=9,
            )
        plt.title(f"Step-count Distribution (total) - {model_name}")
        plt.xlabel("Step Count")
        plt.ylabel("Count")
        plt.xticks(x_vals, step_total.index)
        plt.legend()
        plt.tight_layout()
        p1 = model_output_dir / f"{safe_model_name}_step_distribution_{now}.png"
        fig1.savefig(p1, dpi=150)
        plot_paths.append(p1)
        plt.show()
        plt.close(fig1)

    if len(step_quality):
        fig2, axes = plt.subplots(2, 3, figsize=(18, 8), sharex=True)
        x_step = step_quality.index.to_numpy()
        metric_specs = [
            (0, 0, "avg_score", "Avg Score by Step Count", "Score"),
            (0, 1, "avg_wrong_step", "Avg Wrong Step by Step Count", "Average"),
            (0, 2, "avg_extra_step", "Avg Extra Step by Step Count", "Average"),
            (1, 0, "avg_missing", "Avg Missing by Step Count", "Average"),
            (1, 1, "avg_wrong_order", "Avg Wrong Order by Step Count", "Average"),
            (1, 2, "avg_attr_mismatch", "Avg Attr Mismatch by Step Count", "Average"),
        ]

        for r, c, metric, title, ylabel in metric_specs:
            vals = step_quality[metric].values
            ax = axes[r, c]
            bars = ax.bar(
                x_step,
                vals,
                color="#9ecae1",
                alpha=0.9,
                edgecolor="#4a4a4a",
                linewidth=0.8,
            )
            ax.set_title(title)
            ax.set_ylabel(ylabel)
            for b in bars:
                h = b.get_height()
                ax.text(
                    b.get_x() + b.get_width() / 2,
                    h,
                    f"{h:.2f}",
                    ha="center",
                    va="bottom",
                    fontsize=8,
                )

        for ax in axes[1, :]:
            ax.set_xlabel("Step Count")
        for ax in axes.flat:
            ax.set_xticks(x_step)
            ax.set_xticklabels([str(int(v)) for v in x_step])
            ax.grid(axis="y", alpha=0.25)
        for ax in axes[0, :]:
            ax.tick_params(axis="x", labelbottom=True)

        fig2.suptitle(f"Quality Metrics by Step Count (train+val total) - {model_name}", y=1.02)
        fig2.tight_layout()
        p2 = model_output_dir / f"{safe_model_name}_quality_by_step_count_{now}.png"
        fig2.savefig(p2, dpi=150)
        plot_paths.append(p2)
        plt.show()
        plt.close(fig2)

    if len(df):
        fig3 = plt.figure(figsize=(9, 5))
        score_counts, _, score_patches = plt.hist(
            df["score"],
            bins=20,
            alpha=0.8,
            color="#9ecae1",
            edgecolor="#4a4a4a",
            linewidth=0.8,
            label=f"total (n={len(df)})",
        )
        for h, p in zip(score_counts, score_patches):
            plt.text(
                p.get_x() + p.get_width() / 2,
                h,
                f"{int(round(h))}",
                ha="center",
                va="bottom",
                fontsize=8,
            )
        plt.title(f"Score Distribution (total) - {model_name}")
        plt.xlabel("Score")
        plt.ylabel("Count")
        plt.xlim(0, 100)
        score_xticks = np.arange(0, 101, 5)
        plt.xticks(score_xticks, [str(int(v)) for v in score_xticks])
        plt.legend()
        plt.tight_layout()
        p3 = model_output_dir / f"{safe_model_name}_score_distribution_{now}.png"
        fig3.savefig(p3, dpi=150)
        plot_paths.append(p3)
        plt.show()
        plt.close(fig3)

        fig4 = plt.figure(figsize=(9, 5))
        lat_counts, _, lat_patches = plt.hist(
            df["latency_s"],
            bins=20,
            alpha=0.8,
            color="#9ecae1",
            edgecolor="#4a4a4a",
            linewidth=0.8,
            label=f"total (n={len(df)})",
        )
        for h, p in zip(lat_counts, lat_patches):
            plt.text(
                p.get_x() + p.get_width() / 2,
                h,
                f"{int(round(h))}",
                ha="center",
                va="bottom",
                fontsize=8,
            )
        plt.title(f"Response Time Distribution (total) - {model_name}")
        plt.xlabel("Latency (seconds)")
        plt.ylabel("Count")
        plt.legend()
        plt.tight_layout()
        p4 = model_output_dir / f"{safe_model_name}_latency_distribution_{now}.png"
        fig4.savefig(p4, dpi=150)
        plot_paths.append(p4)
        plt.show()
        plt.close(fig4)

    lines = []
    lines.append("Model Evaluation Report")
    lines.append("=" * 80)
    lines.append(f"Model Name: {model_name}")
    lines.append(f"Base Model: {base_model_name}")
    lines.append(f"Model Directory: {model_dir}")
    lines.append(f"Evaluation Time: {now}")
    lines.append(f"Average Score: {avg_score:.6f}")
    lines.append(f"Average Response Time (s): {avg_time:.6f}")
    lines.append(f"Total Tested Samples: {len(df)}")
    lines.append("")

    if len(df):
        lines.append("Per-split summary:")
        split_summary = df.groupby("split").agg(
            sample_count=("score", "count"),
            avg_score=("score", "mean"),
            avg_latency_s=("latency_s", "mean"),
        )
        lines.append(split_summary.to_string())
        lines.append("")

        lines.append("Step-count sample distribution:")
        lines.append(step_dist.to_string())
        lines.append("")
        lines.append("Step-count total distribution (train+val):")
        lines.append(step_total.to_string())
        lines.append("")
        lines.append("Step-count quality summary (train+val combined):")
        lines.append(step_quality.to_string())
        lines.append("")

        lines.append("Score distribution bins:")
        lines.append(score_hist.to_string())
        lines.append("")

        lines.append("Latency distribution bins:")
        if len(latency_hist):
            for i, c in enumerate(latency_hist):
                lines.append(f"[{latency_edges[i]:.6f}, {latency_edges[i+1]:.6f}) -> {int(c)}")
        lines.append("")

    lines.append("Saved Plots:")
    for p in plot_paths:
        lines.append(str(p.resolve()))
    if terminal_log_path is not None:
        lines.append("")
        lines.append("Terminal Log:")
        lines.append(str(terminal_log_path.resolve()))

    report_path.write_text("\n".join(lines), encoding="utf-8")

    print(f"\nSaved report text: {report_path.resolve()}")
    print(f"Saved detailed csv: {csv_path.resolve()}")
    for p in plot_paths:
        print(f"Saved plot: {p.resolve()}")
    if terminal_log_path is not None:
        print(f"Saved terminal log: {terminal_log_path.resolve()}")


class TeeLogger:
    def __init__(self, *streams):
        self.streams = streams

    def write(self, data):
        for stream in self.streams:
            stream.write(data)
        return len(data)

    def flush(self):
        for stream in self.streams:
            stream.flush()


def build_run_paths(model_dir: str, output_dir: Path):
    output_dir.mkdir(parents=True, exist_ok=True)
    model_name = Path(model_dir).name
    safe_model_name = re.sub(r"[^a-zA-Z0-9._-]+", "_", model_name)
    run_now = datetime.now().strftime("%Y%m%d_%H%M%S")
    model_output_dir = output_dir / model_name
    model_output_dir.mkdir(parents=True, exist_ok=True)
    terminal_log_path = model_output_dir / f"{safe_model_name}_terminal_{run_now}.log"
    return {
        "run_now": run_now,
        "safe_model_name": safe_model_name,
        "model_output_dir": model_output_dir,
        "terminal_log_path": terminal_log_path,
    }


def release_model_from_vram(model=None, tokenizer=None):
    try:
        if model is not None:
            try:
                model.cpu()
            except Exception:
                pass
            del model
        if tokenizer is not None:
            del tokenizer
        gc.collect()
        if torch.cuda.is_available():
            try:
                torch.cuda.synchronize()
            except Exception:
                pass
            torch.cuda.empty_cache()
            try:
                torch.cuda.ipc_collect()
            except Exception:
                pass
    finally:
        plt.close("all")


def main():
    rng = random.Random(SEED)

    model = None
    tokenizer = None
    stdout_origin = sys.stdout
    stderr_origin = sys.stderr
    terminal_fp = None

    try:
        model_dir = resolve_adapter_dir(choose_model_dir())
        base_model_name = resolve_base_model_name(model_dir, BASE_MODEL)

        run_paths = build_run_paths(model_dir, EVAL_DIR)
        model_output_dir = run_paths["model_output_dir"]
        run_now = run_paths["run_now"]
        terminal_log_path = run_paths["terminal_log_path"]

        terminal_fp = open(terminal_log_path, "w", encoding="utf-8", buffering=1)
        sys.stdout = TeeLogger(stdout_origin, terminal_fp)
        sys.stderr = TeeLogger(stderr_origin, terminal_fp)

        print(f"Terminal output is being saved to: {terminal_log_path.resolve()}")
        print(f"Selected model dir: {model_dir}")

        print("Loading model and tokenizer...")
        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name=base_model_name,
            max_seq_length=MAX_SEQ_LENGTH,
            dtype=DTYPE,
            load_in_4bit=LOAD_IN_4BIT,
        )
        try:
            model.load_adapter(model_dir)
        except RuntimeError as e:
            raise RuntimeError(
                "Failed to load LoRA adapter due to model mismatch. "
                f"Adapter dir: {model_dir}. Loaded base model: {base_model_name}. "
                "Please ensure adapter_config.json base_model_name_or_path matches the loaded base model.\n"
                f"Original error: {e}"
            ) from e

        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
        tokenizer.padding_side = "left"

        FastLanguageModel.for_inference(model)
        model.eval()

        print("Loading JSONL dataset...")
        test_records = load_records(TEST_FILE, "test")

        sampled_info = grouped_sampling(test_records, N_PER_STEP_COUNT, rng)
        test_samples = sampled_info["sampled"]
        test_total_dist = sampled_info["total_dist"]
        test_sampled_dist = sampled_info["sampled_dist"]
        test_invalid = sampled_info["invalid_count"]

        print(f"Test records: total={len(test_records)}, invalid_output_format={test_invalid}, sampled={len(test_samples)}")

        print("\nTest step-count totals:")
        print(dict(sorted(test_total_dist.items())))
        print("Test step-count sampled:")
        print(dict(sorted(test_sampled_dist.items())))

        all_samples = test_samples
        rows = []

        print("\nRunning inference and scoring...")
        for i, rec in enumerate(all_samples, start=1):
            pred_text, latency_s = run_autoregressive_inference(model, tokenizer, rec["input"])
            detail = compute_score(rec["output"], pred_text)

            rows.append(
                {
                    "split": rec["split"],
                    "row_id": rec["row_id"],
                    "step_count": rec["step_count"],
                    "score": detail["score"],
                    "latency_s": latency_s,
                    "format_ok": detail["format_ok"],
                    "replace_count": detail["replace_count"],
                    "missing_count": detail["missing_count"],
                    "wrong_order_count": detail["wrong_order_count"],
                    "cost_dur_mismatch_count": detail["cost_dur_mismatch_count"],
                    "extra_count": detail["extra_count"],
                    "prediction": pred_text,
                    "reference": rec["output"],
                }
            )

            if detail["format_ok"]:
                err_info = (
                    f"wrong_step={detail['replace_count']} "
                    f"extra={detail['extra_count']} "
                    f"missing={detail['missing_count']} "
                    f"wrong_order={detail['wrong_order_count']} "
                    f"attr_mismatch={detail['cost_dur_mismatch_count']} "
                    "unparseable=No"
                )
            else:
                err_info = "unparseable=Yes"

            print(
                f"[{i:04d}/{len(all_samples):04d}] row={rec['row_id']} "
                f"steps={rec['step_count']} {err_info} score={detail['score']:.2f} latency={latency_s:.3f}s"
            )

        df = pd.DataFrame(rows)
        summarize_and_save(
            df,
            model_dir=model_dir,
            output_dir=EVAL_DIR,
            base_model_name=base_model_name,
            run_now=run_now,
            model_output_dir=model_output_dir,
            terminal_log_path=terminal_log_path,
        )

        return 0

    finally:
        release_model_from_vram(model, tokenizer)

        if terminal_fp is not None:
            terminal_fp.flush()
            try:
                os.fsync(terminal_fp.fileno())
            except OSError:
                pass

        sys.stdout = stdout_origin
        sys.stderr = stderr_origin

        if terminal_fp is not None:
            terminal_fp.close()

        print("VRAM released and resources cleaned up.")


if __name__ == "__main__":
    os._exit(main())
